# Clase 5 — Modelos de lenguaje e IA generativa

**Módulo de Análisis de Datos de Texto · Notebook 5 de 5 (integrador)**

Cerramos el módulo construyendo un **modelo de lenguaje de n-gramas** en Python puro: el predecesor estadístico de los modelos modernos. Sirve para ver, en miniatura y sin caja negra, qué quiere decir *predecir la siguiente palabra*.

Recorrido:
1. Definición: un modelo de lenguaje como `P(palabra | contexto)`.
2. Modelo de bigramas y trigramas sobre nuestro corpus.
3. Generación autorregresiva con control de **temperatura**.
4. Limitaciones — por qué los Transformers fueron necesarios.
5. Cierre del arco: vectores → significado → contexto → predicción = IA generativa.

## 1. Punto de partida: el corpus

In [1]:
# Corpus de noticias en español (40 documentos cortos, 4 temas).
# Está embebido aquí para que el notebook sea autocontenido.

CORPUS = [
    # --- tecnología ---
    "Una empresa de tecnología presentó un nuevo modelo de inteligencia artificial capaz de generar texto en varios idiomas.",
    "El nuevo procesador promete duplicar la velocidad de los computadores portátiles sin aumentar el consumo de energía.",
    "Los algoritmos de aprendizaje automático están transformando la forma en que las empresas analizan datos masivos.",
    "Un grupo de investigadores publicó un modelo de lenguaje entrenado con miles de millones de palabras en español.",
    "La compañía lanzó una actualización de software que mejora la seguridad y corrige varios errores críticos.",
    "Expertos en ciberseguridad advierten sobre el aumento de ataques con inteligencia artificial generativa.",
    "El nuevo teléfono incorpora una cámara con procesamiento neuronal y reconocimiento de escenas en tiempo real.",
    "Los chips fabricados con tecnología de tres nanómetros llegarán al mercado el próximo año.",
    "Una startup colombiana desarrolló una aplicación que usa modelos de lenguaje para asesoría legal automatizada.",
    "El centro de datos consume menos energía gracias a un sistema de refrigeración basado en agua reciclada.",
    # --- deportes ---
    "El equipo nacional ganó el partido por tres goles a uno en el estadio principal de la capital.",
    "La selección de fútbol clasificó al mundial tras vencer a su rival histórico en tiempo de descuento.",
    "El ciclista colombiano subió al podio en la última etapa de la competencia europea.",
    "Un nuevo récord mundial fue establecido en los cien metros planos durante el campeonato.",
    "El entrenador anunció cambios en la alineación titular para el próximo partido decisivo.",
    "La final del torneo de baloncesto se jugará el sábado entre los dos mejores equipos del país.",
    "El tenista venció en cinco sets a su rival y avanzó a la final del torneo de Grand Slam.",
    "La atleta rompió la marca regional en lanzamiento de jabalina durante los juegos panamericanos.",
    "El nadador colombiano se clasificó a los juegos olímpicos tras una temporada excepcional.",
    "El partido fue suspendido por lluvia y se reanudará mañana en el mismo estadio.",
    # --- política ---
    "El gobierno anunció una nueva reforma tributaria que afectará a las grandes empresas del país.",
    "El congreso aprobó la ley de presupuesto general tras tres jornadas de intenso debate.",
    "La oposición criticó las medidas económicas anunciadas por el ministerio de hacienda esta semana.",
    "El presidente recibió a los líderes regionales para discutir la situación de seguridad en el sur del país.",
    "Las elecciones regionales se celebrarán en octubre con más de cuarenta millones de votantes habilitados.",
    "La corte constitucional declaró inexequible un artículo clave de la última reforma laboral aprobada.",
    "El alcalde presentó el plan de desarrollo con énfasis en movilidad y educación pública.",
    "Un acuerdo de paz entre las facciones políticas permitirá retomar el diálogo sobre la reforma rural.",
    "La ministra de educación defendió ante el congreso el aumento del presupuesto para universidades públicas.",
    "Los partidos políticos comenzaron las negociaciones para definir candidatos a la próxima campaña presidencial.",
    # --- cultura ---
    "La nueva película del director colombiano fue seleccionada para competir en el festival de cine de Cannes.",
    "El museo nacional inauguró una exposición sobre arte precolombino con piezas inéditas del periodo muisca.",
    "Un libro de poesía colombiana ganó el premio internacional de literatura en lengua española este año.",
    "La banda anunció gira mundial con conciertos en quince ciudades de América Latina y Europa.",
    "El festival de cine independiente proyectará más de cien películas durante dos semanas en la capital.",
    "La orquesta sinfónica estrenó una obra del compositor antioqueño con gran acogida del público.",
    "El teatro municipal presenta esta semana una adaptación contemporánea de una obra clásica del siglo de oro.",
    "Una escritora joven publicó su primera novela y ya ha sido traducida a cinco idiomas.",
    "El documental sobre música tradicional del pacífico colombiano fue estrenado en una plataforma de streaming.",
    "La feria del libro recibió más de medio millón de visitantes y cerró con un balance positivo.",
]

LABELS = (
    ["tecnología"] * 10
    + ["deportes"] * 10
    + ["política"] * 10
    + ["cultura"] * 10
)

print(f'Corpus: {len(CORPUS)} documentos en {len(set(LABELS))} categorías.')

Corpus: 40 documentos en 4 categorías.


## 2. Modelo de n-gramas

Un modelo de n-gramas estima `P(palabra | n-1 palabras anteriores)` simplemente **contando** cuántas veces cada secuencia aparece en el corpus.

Para un bigrama:

$$P(w_t \mid w_{t-1}) = \frac{\text{conteo}(w_{t-1}, w_t)}{\text{conteo}(w_{t-1})}$$

In [2]:
import re
from collections import Counter, defaultdict

# Tokenización simple (la misma de la clase 1)
def tokenizar(texto):
    return re.findall(r"[a-záéíóúñü]+", texto.lower())

# Construimos un "documento gigante" pegando todo el corpus
todos_los_tokens = []
for doc in CORPUS:
    todos_los_tokens.extend(["<s>"] + tokenizar(doc) + ["</s>"])

print(f"Total de tokens en el corpus: {len(todos_los_tokens)}")
print("Primeros 15:", todos_los_tokens[:15])

Total de tokens en el corpus: 699
Primeros 15: ['<s>', 'una', 'empresa', 'de', 'tecnología', 'presentó', 'un', 'nuevo', 'modelo', 'de', 'inteligencia', 'artificial', 'capaz', 'de', 'generar']


In [3]:
todos_los_tokens

['<s>',
 'una',
 'empresa',
 'de',
 'tecnología',
 'presentó',
 'un',
 'nuevo',
 'modelo',
 'de',
 'inteligencia',
 'artificial',
 'capaz',
 'de',
 'generar',
 'texto',
 'en',
 'varios',
 'idiomas',
 '</s>',
 '<s>',
 'el',
 'nuevo',
 'procesador',
 'promete',
 'duplicar',
 'la',
 'velocidad',
 'de',
 'los',
 'computadores',
 'portátiles',
 'sin',
 'aumentar',
 'el',
 'consumo',
 'de',
 'energía',
 '</s>',
 '<s>',
 'los',
 'algoritmos',
 'de',
 'aprendizaje',
 'automático',
 'están',
 'transformando',
 'la',
 'forma',
 'en',
 'que',
 'las',
 'empresas',
 'analizan',
 'datos',
 'masivos',
 '</s>',
 '<s>',
 'un',
 'grupo',
 'de',
 'investigadores',
 'publicó',
 'un',
 'modelo',
 'de',
 'lenguaje',
 'entrenado',
 'con',
 'miles',
 'de',
 'millones',
 'de',
 'palabras',
 'en',
 'español',
 '</s>',
 '<s>',
 'la',
 'compañía',
 'lanzó',
 'una',
 'actualización',
 'de',
 'software',
 'que',
 'mejora',
 'la',
 'seguridad',
 'y',
 'corrige',
 'varios',
 'errores',
 'críticos',
 '</s>',
 '<s>',
 

In [13]:
def entrenar_ngramas(tokens, n=2):
    """Devuelve un diccionario contexto -> Counter(palabra_siguiente)."""
    modelo = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        contexto = tuple(tokens[i : i + n - 1])
        siguiente = tokens[i + n - 1]
        modelo[contexto][siguiente] += 1
    return modelo

modelo_bi = entrenar_ngramas(todos_los_tokens, n=2)
modelo_tri = entrenar_ngramas(todos_los_tokens, n=3)
print(f"Bigramas:   {len(modelo_bi)} contextos distintos")
print(f"Trigramas:  {len(modelo_tri)} contextos distintos")

Bigramas:   335 contextos distintos
Trigramas:  581 contextos distintos


In [14]:
# Inspeccionemos qué sigue a "el" en el corpus
print("Top 5 palabras que siguen a 'el':")
for w, c in modelo_bi[("el",)].most_common(5):
    print(f"  {w:15s}  {c}")

Top 5 palabras que siguen a 'el':
  nuevo            2
  aumento          2
  próximo          2
  partido          2
  congreso         2


## 3. Generación autorregresiva

Generar texto es **predecir una palabra, agregarla al contexto, repetir**. La **temperatura** controla cuánta aleatoriedad permitimos:

- *Temperatura baja* (cerca de 0): el modelo siempre elige la palabra más probable. Texto repetitivo, conservador.
- *Temperatura alta* (cerca de 1 o más): el modelo muestrea más libremente. Texto más diverso, también más errático.

In [15]:
import numpy as np

def muestrear(contador, temperatura=1.0):
    """Elige una palabra siguiente dada la distribución, con temperatura."""
    palabras = list(contador.keys())
    conteos = np.array(list(contador.values()), dtype=float)
    # Convertir a probabilidades suavizadas por temperatura
    logits = np.log(conteos + 1e-9) / max(temperatura, 1e-6)
    probs = np.exp(logits - logits.max())
    probs /= probs.sum()
    return np.random.choice(palabras, p=probs)

def generar(modelo, n, max_tokens=25, temperatura=1.0, semilla=None):
    if semilla is not None:
        np.random.seed(semilla)
    contexto = ("<s>",) * (n - 1)
    salida = []
    for _ in range(max_tokens):
        opciones = modelo.get(contexto)
        if not opciones:
            break
        siguiente = muestrear(opciones, temperatura)
        if siguiente == "</s>":
            break
        salida.append(siguiente)
        contexto = tuple(list(contexto[1:]) + [siguiente])
    return " ".join(salida)

In [16]:
modelo_bi

defaultdict(collections.Counter,
            {('<s>',): Counter({'el': 17,
                      'la': 11,
                      'un': 4,
                      'una': 3,
                      'los': 3,
                      'expertos': 1,
                      'las': 1}),
             ('una',): Counter({'obra': 2,
                      'empresa': 1,
                      'actualización': 1,
                      'cámara': 1,
                      'startup': 1,
                      'aplicación': 1,
                      'temporada': 1,
                      'nueva': 1,
                      'exposición': 1,
                      'adaptación': 1,
                      'escritora': 1,
                      'plataforma': 1}),
             ('empresa',): Counter({'de': 1}),
             ('de',): Counter({'la': 3,
                      'lenguaje': 2,
                      'cine': 2,
                      'tecnología': 1,
                      'inteligencia': 1,
                      'generar

In [17]:
generar(modelo_bi, n=2, temperatura=1.0, semilla=1)

'el sur del país'

In [20]:
generar(modelo_tri, n=3, temperatura=1.0, semilla=3)

''

In [7]:
print("=== Trigramas, temperatura BAJA (0.3) — texto repetitivo y conservador ===")
for s in range(3):
    print(f"  · {generar(modelo_tri, n=3, temperatura=0.3, semilla=s)}")

print("\n=== Trigramas, temperatura MEDIA (1.0) ===")
for s in range(3):
    print(f"  · {generar(modelo_tri, n=3, temperatura=1.0, semilla=s)}")

print("\n=== Trigramas, temperatura ALTA (2.0) — más diverso, también más errático ===")
for s in range(3):
    print(f"  · {generar(modelo_tri, n=3, temperatura=2.0, semilla=s)}")

=== Trigramas, temperatura BAJA (0.3) — texto repetitivo y conservador ===
  · 
  · 
  · 

=== Trigramas, temperatura MEDIA (1.0) ===
  · 
  · 
  · 

=== Trigramas, temperatura ALTA (2.0) — más diverso, también más errático ===
  · 
  · 
  · 


## 4. Limitaciones de los n-gramas

Si pruebas a generar textos largos verás dos problemas:

1. **Contexto corto.** Un trigrama solo recuerda las 2 palabras anteriores. No puede saber que estamos hablando de fútbol *desde el principio del párrafo*.
2. **Datos escasos.** Si una secuencia de 3 palabras no aparece *literal* en el corpus, el modelo no sabe qué hacer. Aumentar `n` empeora esto.

**Los Transformers resuelven ambos problemas a la vez:**
- La atención permite mirar TODO el contexto previo, no solo las últimas n-1 palabras.
- Los embeddings hacen que el modelo generalice a secuencias que no vio literalmente, porque palabras parecidas tienen vectores parecidos.

Esa es la diferencia entre nuestro juguete y un LLM moderno como GPT, Claude o Llama.

> **Importante.** El paso de los n-gramas a los Transformers **no cambia la idea fundamental** — sigue siendo `P(palabra | contexto)`. Solo cambian las herramientas para estimar esa probabilidad.

## 5. Mini-proyecto integrador

Usando **todo** lo aprendido en el módulo, vamos a construir un pequeño **clasificador de texto por tema**: dado un texto nuevo, predice si habla de tecnología, deportes, política o cultura.

Reusaremos las representaciones TF-IDF (clase 2) y la similitud coseno (clase 2) — el clasificador es esencialmente un **vecino más cercano** sobre el espacio TF-IDF.

In [ ]:
import nltk; nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

SW = sorted(stopwords.words("spanish"))
tfv = TfidfVectorizer(stop_words=SW, lowercase=True, min_df=2)
Xtfidf = tfv.fit_transform(CORPUS)

def clasificar(texto, k=3):
    q = tfv.transform([texto])
    sims = cosine_similarity(q, Xtfidf).ravel()
    top = np.argsort(-sims)[:k]
    votos = Counter(LABELS[i] for i in top)
    tema, _ = votos.most_common(1)[0]
    return tema, [(LABELS[i], round(sims[i], 3)) for i in top]

ejemplos = [
    "El nuevo algoritmo de aprendizaje profundo supera a los modelos anteriores en precisión.",
    "El defensor anotó un gol espectacular en el minuto noventa del partido.",
    "El proyecto de ley fue debatido durante toda la sesión del congreso.",
    "El director presentó su película más reciente en el festival internacional.",
]
for t in ejemplos:
    tema, vecinos = clasificar(t)
    print(f"  [{tema}]  {t}")
    print(f"     vecinos: {vecinos}\n")

## 6. El arco del módulo, en perspectiva

| Clase | Lo que aprendimos | Para qué nos sirvió hoy |
|------|---|---|
| 1 | Texto → matriz (BoW) | Convertir texto en algo procesable |
| 2 | TF-IDF + similitud coseno | Comparar y clasificar documentos |
| 3 | Embeddings (TruncatedSVD) | Capturar significado por contexto |
| 4 | Atención | Representación contextual de cada palabra |
| 5 | Modelos de lenguaje | Generar texto: `P(palabra \| contexto)` |

**La IA generativa es la consecuencia estadística de todo lo anterior.** Un LLM moderno *es*:
- Embeddings (clase 3) +
- Atención (clase 4) +
- Predicción de la siguiente palabra (clase 5),
escalado a billones de parámetros y entrenado sobre billones de palabras.

Ningún paso fue magia. Todos los pasos fueron estadística sobre texto.

## 7. Ejercicios finales

In [ ]:
# Ejercicio 1: entrena un modelo de 4-gramas (cuatrigramas). ¿El texto generado
# se vuelve más coherente o aparece el problema de "datos escasos"?

# TODO:


In [ ]:
# Ejercicio 2: amplía el clasificador para que muestre la "confianza" de
# la predicción (por ejemplo, qué fracción de los k vecinos están de acuerdo).

# TODO:


In [ ]:
# Ejercicio 3 (reflexión, sin código): ¿qué riesgos identificas al usar un
# LLM moderno (alucinaciones, sesgos, etc.) que se relacionan directamente
# con el hecho de que es un modelo P(palabra | contexto) entrenado sobre
# un corpus específico?

# TODO:


## Cierre del módulo

Gracias por recorrer estas cinco clases. Lo importante no es solo lo que cada técnica hace, sino **por qué surge**: cada paso resuelve una limitación del anterior.

Lo que dejas con ganas de explorar (porque está justo después del horizonte de este módulo): modelos pre-entrenados de Hugging Face, fine-tuning, RAG, evaluación de LLMs. Pero ya tienes el mapa para entender lo que está pasando.